# 07 — Hoàn tất A và H với seed 43–44 trên 2×T4

Notebook chạy **A (FraudGT baseline)** trên GPU 0 và **H (History-Augmented FraudGT)** trên GPU 1. Mỗi tiến trình huấn luyện độc lập hai seed 43 và 44 từ epoch 0.

Cấu hình giữ đúng đợt seed 42: `num_threads=2`, `num_workers=2`, batch 512, fanout `[25,25]`, 256 iterations/epoch và 100 epoch. Kết quả chính dùng threshold cố định 0.50.

In [ ]:
SEED_START = 43
REPEATS = 2  # chạy seed 43 và 44
NUM_THREADS = 2
NUM_WORKERS = 2
print(f'Seeds: {SEED_START}..{SEED_START + REPEATS - 1}')
print('num_threads:', NUM_THREADS, '| num_workers:', NUM_WORKERS)

## 1. Kiểm tra môi trường

In [ ]:
import platform, sys, subprocess, torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB')
if torch.cuda.device_count() < 2:
    print('CẢNH BÁO: chỉ có một GPU; notebook sẽ chạy A rồi H tuần tự.')
subprocess.run(['nvidia-smi'], check=False)

## 2. Cài dependency

In [ ]:
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('PyG wheel index:', wheel_url)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'torchmetrics', 'yacs', 'datatable',
                'pandas', 'matplotlib', 'wandb', 'ogb', 'tensorboardX',
                'pyyaml'], check=True)
print('Dependencies installed.')

## 3. Lấy repository và ghi commit

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/mhiunguyen/TH-FraudGT.git'
repo = Path('/kaggle/working/TH-FraudGT')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository:', repo)
print('Commit:', commit)
required = [
    repo / 'fraudGT' / 'datasets' / 'history_features.py',
    repo / 'configs' / 'AML-Small-HI' / 'AML-Small-HI-History-T4.yaml',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError('Repository thiếu file cần thiết: ' + str(missing))

## 4. Gắn dữ liệu AML-Small-HI

In [ ]:
from shutil import copy2

candidates = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if not candidates:
    raise FileNotFoundError('Hãy Add Input bộ IBM AML; không tìm thấy HI-Small_Trans.csv.')
source = candidates[0]
destination = repo / 'data' / 'AML' / 'HI-Small_Trans.csv'
destination.parent.mkdir(parents=True, exist_ok=True)
if not destination.exists() or destination.stat().st_size != source.stat().st_size:
    copy2(source, destination)
print('Dataset:', destination)
print(f'Size: {destination.stat().st_size / 1024**2:.1f} MiB')

## 5. Sinh config A/H cho seed 43–44

In [ ]:
import copy, yaml

cfg_root = repo / 'configs' / 'AML-Small-HI'
h_cfg = yaml.safe_load((cfg_root / 'AML-Small-HI-History-T4.yaml').read_text(encoding='utf-8'))
# Sinh A trực tiếp từ H rồi tắt history: bảo đảm hai cấu hình chỉ khác add_history.
a_cfg = copy.deepcopy(h_cfg)
thresholds = [round(x * 0.05, 2) for x in range(1, 20)]
for cfg in [a_cfg, h_cfg]:
    cfg['out_dir'] = str(repo / 'results')
    cfg['dataset']['dir'] = str(repo / 'data')
    cfg['seed'] = SEED_START
    cfg['num_threads'] = NUM_THREADS
    cfg['num_workers'] = NUM_WORKERS
    cfg['train']['enable_ckpt'] = False
    cfg['wandb']['use'] = False
    cfg.setdefault('mvia', {})['thresholds'] = thresholds

a_cfg['dataset']['add_history'] = False
a_cfg['gt']['head'] = 'hetero_edge'
h_cfg['dataset']['add_history'] = True
h_cfg['gt']['head'] = 'hetero_edge'

# Khóa các thông số công bằng quan trọng giữa A và H.
for key in ['num_threads', 'num_workers']:
    assert a_cfg[key] == h_cfg[key]
for key in ['neighbor_sizes', 'iter_per_epoch', 'batch_size', 'eval_period']:
    assert a_cfg['train'][key] == h_cfg['train'][key], key
for key in ['batch_accumulation', 'base_lr', 'max_epoch', 'scheduler']:
    assert a_cfg['optim'][key] == h_cfg['optim'][key], key

generated = Path('/kaggle/working/generated_configs')
generated.mkdir(parents=True, exist_ok=True)
A_CFG = generated / 'AML-Small-HI-A-Seeds43-44.yaml'
H_CFG = generated / 'AML-Small-HI-H-Seeds43-44.yaml'
A_CFG.write_text(yaml.safe_dump(a_cfg, sort_keys=False), encoding='utf-8')
H_CFG.write_text(yaml.safe_dump(h_cfg, sort_keys=False), encoding='utf-8')
print('A config:', A_CFG)
print('H config:', H_CFG)
print('A add_history:', a_cfg['dataset']['add_history'])
print('H add_history:', h_cfg['dataset']['add_history'])
print('Threads/workers:', NUM_THREADS, NUM_WORKERS)
print('Epochs:', a_cfg['optim']['max_epoch'], '| repeats:', REPEATS)

## 6. Tạo cache tuần tự trước khi chạy song song

A và H được tiền xử lý lần lượt để tránh hai tiến trình cùng ghi cache. Cache H tách riêng khỏi cache A.

In [ ]:
import gc, time
sys.path.insert(0, str(repo))
from fraudGT.datasets.aml_dataset import AMLDataset

for name, add_history in [('A', False), ('H', True)]:
    started = time.time()
    cache = AMLDataset(root=str(repo / 'data' / 'AML'), name='Small-HI',
                       reverse_mp=True, add_ports=True, add_history=add_history)
    print(name, 'processed files:', cache.processed_paths)
    print(f'{name} cache ready in {(time.time() - started) / 60:.1f} minutes')
    del cache
    gc.collect()

## 7. Huấn luyện A và H song song

Trên 2×T4: A dùng GPU 0, H dùng GPU 1. Nếu chỉ có một GPU, notebook tự chạy tuần tự. Heartbeat xuất hiện mỗi phút; log chi tiết nằm trong `/kaggle/working`.

In [ ]:
import time

jobs = [
    {'name': 'A', 'cfg': A_CFG, 'gpu': 0, 'tag': 'A-Seeds43-44',
     'log': Path('/kaggle/working/A_seeds43_44.log')},
    {'name': 'H', 'cfg': H_CFG, 'gpu': 1 if torch.cuda.device_count() >= 2 else 0,
     'tag': 'H-Seeds43-44', 'log': Path('/kaggle/working/H_seeds43_44.log')},
]

def command(job):
    return [sys.executable, '-u', '-m', 'fraudGT.main', '--cfg', str(job['cfg']),
            '--repeat', str(REPEATS), '--gpu', str(job['gpu']),
            'name_tag', job['tag']]

def run_jobs(jobs_to_run):
    handles = []
    for job in jobs_to_run:
        stream = job['log'].open('w', encoding='utf-8')
        process = subprocess.Popen(command(job), cwd=repo, stdout=stream,
                                   stderr=subprocess.STDOUT, text=True)
        handles.append((job, process, stream))
        print(f"Started {job['name']} on GPU {job['gpu']} — PID {process.pid}")
    started = time.time()
    while any(process.poll() is None for _, process, _ in handles):
        time.sleep(60)
        states = ', '.join(
            f"{job['name']}={'running' if process.poll() is None else 'done'}"
            for job, process, _ in handles)
        print(f'[heartbeat] {(time.time() - started) / 60:.0f} min | {states}', flush=True)
        subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,utilization.gpu',
                        '--format=csv,noheader'], check=False)
    failures = []
    for job, process, stream in handles:
        stream.close()
        if process.returncode != 0:
            failures.append(job['name'])
            tail = job['log'].read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
            print(f"\n===== {job['name']} FAILED =====")
            print('\n'.join(tail))
    if failures:
        raise RuntimeError('Failed jobs: ' + ', '.join(failures))

if torch.cuda.device_count() >= 2:
    run_jobs(jobs)
else:
    for job in jobs:
        run_jobs([job])
print('A và H đã hoàn tất seed 43–44.')

## 8. Kiểm tra log và xác nhận seed độc lập

In [ ]:
for job in jobs:
    text = job['log'].read_text(encoding='utf-8', errors='replace')
    assert 'Run ID 43: seed=43' in text, f"{job['name']} thiếu seed 43"
    assert 'Run ID 44: seed=44' in text, f"{job['name']} thiếu seed 44"
    print(f"===== {job['name']} LOG TAIL =====")
    print('\n'.join(text.splitlines()[-35:]))

## 9. Tổng hợp kết quả

Bảng chính dùng threshold cố định 0.50. Bảng validation-selected là phân tích phụ; test không tham gia chọn epoch hoặc threshold.

In [ ]:
import pandas as pd

A_DIR = repo / 'results' / f'{A_CFG.stem}-A-Seeds43-44-gpu0'
H_GPU = 1 if torch.cuda.device_count() >= 2 else 0
H_DIR = repo / 'results' / f'{H_CFG.stem}-H-Seeds43-44-gpu{H_GPU}'
summarizer = repo / 'scripts' / 'summarize_thresholds.py'
frames = []
summary_files = []
for name, run_dir in [('A', A_DIR), ('H', H_DIR)]:
    fixed = Path(f'/kaggle/working/summary_{name}_seeds43_44_fixed_050.csv')
    selected = Path(f'/kaggle/working/summary_{name}_seeds43_44_val_selected.csv')
    subprocess.run([sys.executable, str(summarizer), str(run_dir),
                    '--output', str(fixed), '--fixed-threshold', '0.50'], check=True)
    subprocess.run([sys.executable, str(summarizer), str(run_dir),
                    '--output', str(selected)], check=True)
    summary_files.extend([fixed, selected])
    for protocol, path in [('fixed_0.50_primary', fixed),
                           ('validation_selected_secondary', selected)]:
        frame = pd.read_csv(path)
        frame.insert(0, 'model', name)
        frame.insert(1, 'protocol', protocol)
        frames.append(frame)

results = pd.concat(frames, ignore_index=True)
RESULTS_PATH = Path('/kaggle/working/summary_AH_seeds43_44.csv')
results.to_csv(RESULTS_PATH, index=False)
metrics = ['val_f1', 'test_f1', 'test_precision', 'test_recall', 'test_auc']
aggregate = results.groupby(['model', 'protocol'])[metrics].agg(['mean', 'std']).round(5)
AGG_PATH = Path('/kaggle/working/summary_AH_seeds43_44_mean_std.csv')
aggregate.to_csv(AGG_PATH)
display(results[['model', 'protocol', 'seed', 'best_epoch', 'threshold',
                 'val_f1', 'test_f1', 'test_precision', 'test_recall', 'test_auc']])
display(aggregate)
print('Lưu ý: đây mới là seed 43–44; ghép seed 42 sau khi tải ZIP về.')
print('Detailed:', RESULTS_PATH)
print('Mean/std 43–44:', AGG_PATH)

## 10. Vẽ so sánh hai seed mới

In [ ]:
import matplotlib.pyplot as plt

primary = results[results['protocol'] == 'fixed_0.50_primary'].copy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric, title in zip(
        axes, ['test_f1', 'test_precision', 'test_recall'],
        ['Test F1 @ 0.50', 'Test precision @ 0.50', 'Test recall @ 0.50']):
    for model, color in [('A', '#2563eb'), ('H', '#16a34a')]:
        part = primary[primary['model'] == model].sort_values('seed')
        ax.plot(part['seed'], part[metric], marker='o', linewidth=2,
                label=model, color=color)
    ax.set_xticks([43, 44]); ax.set_xlabel('Seed'); ax.set_title(title)
    ax.grid(alpha=0.25); ax.legend()
fig.tight_layout()
PLOT_PATH = Path('/kaggle/working/comparison_AH_seeds43_44.png')
fig.savefig(PLOT_PATH, dpi=180, bbox_inches='tight')
plt.show()
print('Plot:', PLOT_PATH)

## 11. Đóng gói để tải về

In [ ]:
import shutil

bundle = Path('/kaggle/working/AH_seeds43_44_T4x2_artifacts')
bundle.mkdir(exist_ok=True)
files = [A_CFG, H_CFG, RESULTS_PATH, AGG_PATH, PLOT_PATH] + summary_files
files += [job['log'] for job in jobs]
for path in files:
    if path.exists():
        shutil.copy2(path, bundle / path.name)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('Download:', archive)
print('Sau khi tải về, giữ cả ZIP này và bộ artifact seed 42 để ghép thành bảng 3 seed.')